# ABLATION STUDY — Kontribusi Sentimen Berita (X1 vs X1+X2)
## Prediksi Volatilitas Return Saham Sektor Energi LQ45

**Tesis**: Pemodelan Prediksi Volatilitas Return Saham Sektor Energi (LQ45) Berdasarkan Analisis Sentimen Menggunakan Metode XGBoost dan LSTM

**Penulis**: Ilda Nurida (NRP 6032251068) | **Pembimbing**: Dr. Ir. Ahmad Saikhu, S.Si, MT.

---

### Tujuan Notebook
Menguji secara **statistik** apakah penambahan fitur sentimen (X2: `Sentiment_Mean`, `Sentiment_Dispersion`, `News_Count`, `Ada_Berita`) benar-benar **meningkatkan akurasi** prediksi volatilitas dibanding hanya memakai fitur teknikal (X1) saja — bukan sekadar melihat *feature importance* SHAP (yang cuma menunjukkan kontribusi relatif DI DALAM satu model, bukan bukti bahwa model lebih akurat KARENA sentimen).

### Desain Ablation
| Varian | Fitur | Hyperparameter |
|---|---|---|
| **X1 saja** | 13 fitur teknikal | **Sama persis** dengan varian X1+X2 |
| **X1 + X2** | 13 fitur teknikal + 4 fitur sentimen | **Sama persis** dengan varian X1 saja |

Arsitektur/hyperparameter **dikunci identik** di kedua varian — supaya selisih hasil murni disebabkan oleh ada/tidaknya sentimen, bukan tercampur efek tuning berbeda.

### Uji Signifikansi: Diebold-Mariano Test
Selisih RMSE/MAPE mentah **tidak cukup** untuk mengklaim "signifikan" — perlu uji statistik formal. Diebold-Mariano (1995) menguji H0: kedua model punya akurasi forecast yang **sama** (tidak berbeda signifikan), berdasarkan selisih loss (squared error) tiap titik prediksi, bukan cuma rata-rata agregat.

$$DM = \frac{\bar{d}}{\sqrt{\widehat{Var}(\bar{d})}}, \quad d_t = e_{1,t}^2 - e_{2,t}^2$$

Kalau `p-value < 0.05`, selisih akurasi **signifikan secara statistik** — barulah klaim "sentimen berkontribusi signifikan" bisa dipertanggungjawabkan di abstrak/Bab 4-5.

---
## CELL 1 — Import, Konfigurasi, Definisi Fitur X1 vs X1+X2

In [1]:
import os, warnings, json as _json
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from scipy.stats import norm

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

ROOT = Path(os.getcwd())
for _ in range(6):
    if (ROOT / '03_dataset').exists(): break
    ROOT = ROOT.parent
if not (ROOT / '03_dataset').exists():
    ROOT = Path(r'e:\DATA PENTING ILDA NURIDA\DOKUMEN\KULIAH\S2 2025\THESIS'
                r'\Penelitian\TESIS_PEMODELAN PREDIKSI VOLATILITAS RETURN SAHAM_2026')

DIR_FINAL  = ROOT / '03_dataset' / 'final'
DIR_TABEL  = ROOT / '06_dokumen_tesis' / 'tabel'
DIR_GAMBAR = ROOT / '06_dokumen_tesis' / 'gambar' / 'model'
for d in (DIR_TABEL, DIR_GAMBAR):
    d.mkdir(parents=True, exist_ok=True)

URUTAN_EMITEN = ['ADRO', 'ADMR', 'ITMG', 'PTBA', 'MEDC', 'AKRA', 'PGAS']
KOLOM_Y  = 'Volatilitas_Target'
LOOKBACK = 10

KOLOM_X1 = [
    'Log_Return', 'Vol_Lag1', 'Vol_Lag2', 'Vol_Lag3', 'Vol_MA5', 'Vol_STD5',
    'MA5_Ratio', 'MA10_Ratio', 'MA20_Ratio', 'RSI14',
    'Stoch_K', 'Stoch_D', 'Volume_ZScore',
]
KOLOM_X2 = ['Sentiment_Mean', 'Sentiment_Dispersion', 'News_Count', 'Ada_Berita']

# ── 2 varian fitur yang diuji (ablation) ──
VARIAN_FITUR = {
    'X1_saja': KOLOM_X1,
    'X1_X2':   KOLOM_X1 + KOLOM_X2,
}

# ── Hyperparameter DIKUNCI SAMA utk kedua varian (prinsip ablation yang benar) ──
XGB_PARAMS = dict(
    n_estimators=300, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, random_state=SEED,
)
LSTM_UNIT1, LSTM_UNIT2, LSTM_DROPOUT, LSTM_LR = 32, 16, 0.25, 1e-4
EPOCHS, BATCH, PATIENCE = 100, 16, 20

print('=' * 75)
print('  ABLATION STUDY -- KONFIGURASI SIAP')
print('=' * 75)
print(f'ROOT        : {ROOT}')
print(f'Varian fitur: X1_saja ({len(KOLOM_X1)} fitur) vs X1_X2 ({len(KOLOM_X1+KOLOM_X2)} fitur)')
print(f'XGBoost params (dikunci sama)  : {XGB_PARAMS}')
print(f'LSTM arsitektur (dikunci sama) : LSTM({LSTM_UNIT1})->LSTM({LSTM_UNIT2}), '
      f'dropout={LSTM_DROPOUT}, lr={LSTM_LR}')
print('=' * 75)


  ABLATION STUDY -- KONFIGURASI SIAP
ROOT        : e:\DATA PENTING ILDA NURIDA\DOKUMEN\KULIAH\S2 2025\THESIS\Penelitian\TESIS_PEMODELAN PREDIKSI VOLATILITAS RETURN SAHAM_2026
Varian fitur: X1_saja (13 fitur) vs X1_X2 (17 fitur)
XGBoost params (dikunci sama)  : {'n_estimators': 300, 'max_depth': 4, 'learning_rate': 0.05, 'subsample': 0.8, 'colsample_bytree': 0.8, 'random_state': 42}
LSTM arsitektur (dikunci sama) : LSTM(32)->LSTM(16), dropout=0.25, lr=0.0001


---
## CELL 2 — Fungsi Bantu: Metrik, Sekuens LSTM, Diebold-Mariano Test

In [2]:
def evaluasi_semua_metrik(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float); y_pred = np.asarray(y_pred, dtype=float)
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae  = float(mean_absolute_error(y_true, y_pred))
    mask = y_true != 0
    mape = float(np.mean(np.abs((y_true[mask]-y_pred[mask])/y_true[mask]))*100) if mask.sum()>0 else np.nan
    r2 = float(r2_score(y_true, y_pred)) if len(y_true) > 1 else np.nan
    return {'RMSE': rmse, 'MAE': mae, 'MAPE (%)': mape, 'R2': r2}


def buat_sekuens(X, y, lookback):
    Xs, ys = [], []
    for i in range(lookback, len(X)):
        Xs.append(X[i-lookback:i]); ys.append(y[i])
    return np.array(Xs), np.array(ys)


def buat_model_lstm(n_fitur):
    model = Sequential()
    model.add(Input(shape=(LOOKBACK, n_fitur)))
    model.add(LSTM(LSTM_UNIT1, return_sequences=True, recurrent_dropout=0.1))
    model.add(Dropout(LSTM_DROPOUT))
    model.add(LSTM(LSTM_UNIT2, return_sequences=False, recurrent_dropout=0.1))
    model.add(Dropout(LSTM_DROPOUT))
    model.add(Dense(16, activation='relu'))
    model.add(Dense(1, activation='softplus'))
    model.compile(optimizer=Adam(learning_rate=LSTM_LR), loss='mse', metrics=['mae'])
    return model


def diebold_mariano_test(y_true, pred1, pred2, h=1):
    """DM test (Diebold & Mariano, 1995). H0: akurasi forecast model 1 & 2 SAMA.
    d_t = e1_t^2 - e2_t^2 (loss differential, squared error).
    p-value < 0.05 -> selisih akurasi SIGNIFIKAN secara statistik."""
    y_true = np.asarray(y_true, dtype=float)
    e1 = y_true - np.asarray(pred1, dtype=float)
    e2 = y_true - np.asarray(pred2, dtype=float)
    d = e1**2 - e2**2
    n = len(d)
    mean_d = d.mean()

    gamma0 = np.var(d, ddof=0)
    var_d = gamma0
    for lag in range(1, h):
        if n - lag > 1:
            cov = np.cov(d[:-lag], d[lag:])[0, 1]
            var_d += 2 * cov
    var_d = max(var_d, 1e-12)  # hindari div-by-zero

    dm_stat = mean_d / np.sqrt(var_d / n)
    p_value = 2 * (1 - norm.cdf(abs(dm_stat)))
    return dm_stat, p_value


print('Fungsi evaluasi_semua_metrik, buat_sekuens, buat_model_lstm, diebold_mariano_test siap.')


Fungsi evaluasi_semua_metrik, buat_sekuens, buat_model_lstm, diebold_mariano_test siap.


---
## CELL 3 — Load Dataset Train/Test per Emiten

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# Inisialisasi wadah hasil per emiten
dict_train, dict_test, dict_scaler = {}, {}, {}

for emiten in URUTAN_EMITEN:
    # Ambil data emiten ini, urutkan kronologis
    csv_files = sorted(DIR_FINAL.glob('*.csv'))
    if len(csv_files) == 0:
        raise FileNotFoundError(f'Tidak ditemukan file CSV di {DIR_FINAL}')

    if len(csv_files) == 1:
        data_path = csv_files[0]
    else:
        candidates = [p for p in csv_files if 'final' in p.name.lower() or 'all' in p.name.lower() or 'dataset' in p.name.lower()]
        if len(candidates) == 1:
            data_path = candidates[0]
        else:
            raise FileNotFoundError(
                f'Tidak bisa menentukan file dataset final di {DIR_FINAL}. '
                f'Pilihan: {[p.name for p in csv_files]}'
            )

    df = pd.read_csv(data_path, parse_dates=['Tanggal'])
    print(f'Memuat data dari: {data_path.name} (rows={len(df)})')

    # Inisialisasi wadah hasil per emiten
    dict_train, dict_test, dict_scaler = {}, {}, {}

    for emiten in URUTAN_EMITEN:
        d = df[df['Kode_Emiten'] == emiten].copy()
        d = d.sort_values('Tanggal').reset_index(drop=True)

        n_train = int(len(d) * 0.8)
        train = d.iloc[:n_train].copy()
        test  = d.iloc[n_train:].copy()

        dict_train[emiten] = train
        dict_test[emiten]  = test

        print(f'{emiten}: latih={len(train)}, uji={len(test)}')

    print('\n✅ Data per emiten siap.')
    d = d.sort_values('Tanggal').reset_index(drop=True)

    # Split 80% latih : 20% uji secara kronologis (tanpa acak)
    n_train = int(len(d) * 0.8)
    train = d.iloc[:n_train].copy()
    test  = d.iloc[n_train:].copy()

    # Scaling (fit HANYA pada data latih untuk mencegah data leakage)
    scaler = MinMaxScaler()
    train_scaled = scaler.fit_transform(train[kolom_x])
    test_scaled  = scaler.transform(test[kolom_x])

    # Simpan
    dict_train[emiten]  = (train_scaled, train[kolom_y].values)
    dict_test[emiten]   = (test_scaled,  test[kolom_y].values)
    dict_scaler[emiten] = scaler

    print(f'{emiten}: latih={len(train)}, uji={len(test)}')

print('\n✅ Data per emiten siap.')

NameError: name 'df' is not defined

---
## CELL 4 — Training & Evaluasi XGBoost: X1 saja vs X1+X2

Hyperparameter **dikunci sama** (`XGB_PARAMS`) untuk kedua varian — tidak pakai `RandomizedSearchCV` di sini supaya perbandingan murni soal fitur, bukan tercampur efek tuning yang beda antar varian.

In [ ]:
print('=' * 75)
print('  CELL 4 -- TRAINING XGBOOST: X1_saja vs X1_X2')
print('=' * 75)

hasil_xgb_ablation = {v: {} for v in VARIAN_FITUR}

for emiten in dict_train:
    df_tr, df_te = dict_train[emiten], dict_test[emiten]

    for nama_varian, kolom_x in VARIAN_FITUR.items():
        X_tr, y_tr = df_tr[kolom_x].values, df_tr[KOLOM_Y].values * 100
        X_te, y_te = df_te[kolom_x].values, df_te[KOLOM_Y].values

        model = xgb.XGBRegressor(**XGB_PARAMS)
        model.fit(X_tr, y_tr)

        y_pred = np.clip(model.predict(X_te) / 100, 0, None)
        metrik = evaluasi_semua_metrik(y_te, y_pred)

        hasil_xgb_ablation[nama_varian][emiten] = {
            'y_true': y_te, 'y_pred': y_pred, **metrik,
        }
        print(f'  XGBoost | {emiten:6s} | {nama_varian:8s}: RMSE={metrik["RMSE"]:.5f} | '
              f'MAPE={metrik["MAPE (%)"]:.2f}% | R2={metrik["R2"]:.4f}')

print('\nTraining XGBoost (2 varian) selesai.')


---
## CELL 5 — Training & Evaluasi LSTM: X1 saja vs X1+X2

Arsitektur **dikunci sama** untuk kedua varian (`LSTM_UNIT1/2`, dropout, lr) — hanya jumlah fitur input (dimensi ke-3 sequence) yang berbeda.

In [ ]:
print('=' * 75)
print('  CELL 5 -- TRAINING LSTM: X1_saja vs X1_X2')
print('=' * 75)

hasil_lstm_ablation = {v: {} for v in VARIAN_FITUR}

for emiten in dict_train:
    df_tr, df_te = dict_train[emiten], dict_test[emiten]

    for nama_varian, kolom_x in VARIAN_FITUR.items():
        X_tr_raw, y_tr_raw = df_tr[kolom_x].values, df_tr[KOLOM_Y].values
        X_te_raw, y_te_raw = df_te[kolom_x].values, df_te[KOLOM_Y].values

        Xs_tr, ys_tr = buat_sekuens(X_tr_raw, y_tr_raw, LOOKBACK)
        Xs_te, ys_te = buat_sekuens(X_te_raw, y_te_raw, LOOKBACK)

        if len(Xs_tr) < 20 or len(Xs_te) < 5:
            print(f'  WARNING {emiten}/{nama_varian}: data sekuens terlalu sedikit, skip')
            continue

        tf.keras.backend.clear_session()
        tf.random.set_seed(SEED)
        model = buat_model_lstm(len(kolom_x))

        cbs = [
            EarlyStopping(monitor='val_loss', patience=PATIENCE, restore_best_weights=True, verbose=0),
            ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=7, min_lr=1e-6, verbose=0),
        ]
        model.fit(Xs_tr, ys_tr, epochs=EPOCHS, batch_size=BATCH, validation_split=0.15,
                  callbacks=cbs, verbose=0, shuffle=False)

        y_pred = np.clip(model.predict(Xs_te, verbose=0).flatten(), 0, None)
        metrik = evaluasi_semua_metrik(ys_te, y_pred)

        hasil_lstm_ablation[nama_varian][emiten] = {
            'y_true': ys_te, 'y_pred': y_pred, **metrik,
        }
        print(f'  LSTM    | {emiten:6s} | {nama_varian:8s}: RMSE={metrik["RMSE"]:.5f} | '
              f'MAPE={metrik["MAPE (%)"]:.2f}% | R2={metrik["R2"]:.4f}')

print('\nTraining LSTM (2 varian) selesai.')


---
## CELL 6 — Tabel Perbandingan Ablation (-> Tabel 4.X)

In [ ]:
print('=' * 90)
print('  TABEL ABLATION -- X1 SAJA vs X1+X2 (XGBOOST & LSTM)')
print('=' * 90)

baris = []
for emiten in dict_train:
    if emiten not in hasil_xgb_ablation['X1_saja'] or emiten not in hasil_xgb_ablation['X1_X2']:
        continue
    xgb_x1  = hasil_xgb_ablation['X1_saja'][emiten]
    xgb_x1x2 = hasil_xgb_ablation['X1_X2'][emiten]
    lstm_x1  = hasil_lstm_ablation['X1_saja'].get(emiten)
    lstm_x1x2 = hasil_lstm_ablation['X1_X2'].get(emiten)

    baris.append({
        'Emiten': emiten,
        'XGB_RMSE_X1': round(xgb_x1['RMSE'], 5), 'XGB_RMSE_X1X2': round(xgb_x1x2['RMSE'], 5),
        'XGB_Delta(%)': round((xgb_x1['RMSE']-xgb_x1x2['RMSE'])/xgb_x1['RMSE']*100, 2),
        'LSTM_RMSE_X1': round(lstm_x1['RMSE'], 5) if lstm_x1 else None,
        'LSTM_RMSE_X1X2': round(lstm_x1x2['RMSE'], 5) if lstm_x1x2 else None,
        'LSTM_Delta(%)': round((lstm_x1['RMSE']-lstm_x1x2['RMSE'])/lstm_x1['RMSE']*100, 2) if lstm_x1 and lstm_x1x2 else None,
    })

df_ablation = pd.DataFrame(baris)
pd.set_option('display.width', 200, 'display.max_columns', None)
print(df_ablation.to_string(index=False))
print()
print('Delta(%) positif = X1+X2 LEBIH BAIK (RMSE lebih rendah) drpd X1 saja')

path_ablation = DIR_TABEL / 'tabel_ablation_sentimen.csv'
df_ablation.to_csv(path_ablation, index=False, encoding='utf-8-sig')
print(f'\nTabel disimpan: {path_ablation}')


---
## CELL 7 — Uji Signifikansi Statistik: Diebold-Mariano Test

**Bagian paling penting notebook ini** — menjawab langsung apakah selisih akurasi X1 vs X1+X2 signifikan secara statistik (bukan cuma kebetulan variasi sampel).

In [ ]:
print('=' * 90)
print('  DIEBOLD-MARIANO TEST -- X1 SAJA vs X1+X2')
print('=' * 90)

baris_dm = []
for emiten in dict_train:
    if emiten not in hasil_xgb_ablation['X1_saja'] or emiten not in hasil_xgb_ablation['X1_X2']:
        continue

    # -- XGBoost --
    y_true_xgb = hasil_xgb_ablation['X1_saja'][emiten]['y_true']
    pred_x1_xgb  = hasil_xgb_ablation['X1_saja'][emiten]['y_pred']
    pred_x1x2_xgb = hasil_xgb_ablation['X1_X2'][emiten]['y_pred']
    dm_xgb, p_xgb = diebold_mariano_test(y_true_xgb, pred_x1_xgb, pred_x1x2_xgb)

    row = {
        'Emiten': emiten,
        'DM_Stat_XGB': round(dm_xgb, 4), 'p_value_XGB': round(p_xgb, 4),
        'Signifikan_XGB (p<0.05)': 'YA' if p_xgb < 0.05 else 'Tidak',
    }

    # -- LSTM (kalau tersedia) --
    if emiten in hasil_lstm_ablation['X1_saja'] and emiten in hasil_lstm_ablation['X1_X2']:
        y_true_lstm = hasil_lstm_ablation['X1_saja'][emiten]['y_true']
        pred_x1_lstm = hasil_lstm_ablation['X1_saja'][emiten]['y_pred']
        pred_x1x2_lstm = hasil_lstm_ablation['X1_X2'][emiten]['y_pred']
        dm_lstm, p_lstm = diebold_mariano_test(y_true_lstm, pred_x1_lstm, pred_x1x2_lstm)
        row.update({
            'DM_Stat_LSTM': round(dm_lstm, 4), 'p_value_LSTM': round(p_lstm, 4),
            'Signifikan_LSTM (p<0.05)': 'YA' if p_lstm < 0.05 else 'Tidak',
        })

    baris_dm.append(row)
    print(f"  {emiten:6s}: DM_XGB={dm_xgb:+.3f} (p={p_xgb:.4f}, "
          f"{'SIGNIFIKAN' if p_xgb<0.05 else 'tidak signifikan'})"
          + (f" | DM_LSTM={dm_lstm:+.3f} (p={p_lstm:.4f}, "
             f"{'SIGNIFIKAN' if p_lstm<0.05 else 'tidak signifikan'})" if 'dm_lstm' in dir() else ""))

df_dm = pd.DataFrame(baris_dm)
print()
print(df_dm.to_string(index=False))

path_dm = DIR_TABEL / 'tabel_diebold_mariano_ablation.csv'
df_dm.to_csv(path_dm, index=False, encoding='utf-8-sig')
print(f'\nTabel disimpan: {path_dm}')

n_sig_xgb = (df_dm['Signifikan_XGB (p<0.05)'] == 'YA').sum()
print(f'\nRingkasan: XGBoost signifikan di {n_sig_xgb}/{len(df_dm)} emiten')
if 'Signifikan_LSTM (p<0.05)' in df_dm.columns:
    n_sig_lstm = (df_dm['Signifikan_LSTM (p<0.05)'] == 'YA').sum()
    print(f'           LSTM signifikan di {n_sig_lstm}/{len(df_dm)} emiten')

print()
print('INTERPRETASI DM_Stat:')
print('  DM_Stat NEGATIF -> X1+X2 (Model 2) errornya LEBIH KECIL -> sentimen MEMBANTU')
print('  DM_Stat POSITIF -> X1 saja (Model 1) errornya LEBIH KECIL -> sentimen TIDAK membantu/merugikan')
print('  p-value < 0.05  -> selisih tsb SIGNIFIKAN secara statistik (bukan kebetulan)')


In [ ]:
print(df_ablation.columns.tolist())
print(df_ablation.head())

---
## CELL 8 — Visualisasi Perbandingan RMSE: X1 vs X1+X2 (-> Gambar 4.X)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

print('=' * 70)
print('  VISUALISASI ABLATION STUDY (TERPISAH, 1 KOLOM ITS)')
print('=' * 70)

# Font Times New Roman 8pt sesuai template ITS
plt.rcParams.update({
    'font.family'    : 'Times New Roman',
    'font.size'      : 8,
    'axes.titlesize' : 8,
    'axes.labelsize' : 8,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 8,
})

if 'df_ablation' not in globals():
    path_ablation = DIR_TABEL / 'tabel_ablation_sentimen.csv'
    if not path_ablation.exists():
        raise FileNotFoundError(f'{path_ablation} not found. Jalankan kembali CELL 12 terlebih dahulu.')
    df_ablation = pd.read_csv(path_ablation, encoding='utf-8-sig')

emiten_list = list(df_ablation['Emiten'])
x = np.arange(len(emiten_list)); w = 0.35

# ============================================================
# GAMBAR 1 — XGBoost (file terpisah)
# ============================================================
fig, ax = plt.subplots(figsize=(3.5, 2.6))
import matplotlib.pyplot as plt

print('=' * 70)
print('  VISUALISASI ABLATION STUDY (TERPISAH, 1 KOLOM ITS)')
print('=' * 70)

# Font Times New Roman 8pt sesuai template ITS
plt.rcParams.update({
    'font.family'    : 'Times New Roman',
    'font.size'      : 8,
    'axes.titlesize' : 8,
    'axes.labelsize' : 8,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 8,
})

if 'df_ablation' not in globals():
    path_ablation = DIR_TABEL / 'tabel_ablation_sentimen.csv'
    if not path_ablation.exists():
        raise FileNotFoundError(f'{path_ablation} not found. Jalankan kembali CELL 12 terlebih dahulu.')
    df_ablation = pd.read_csv(path_ablation, encoding='utf-8-sig')

required_cols = {'Model', 'Emiten', 'RMSE_Tanpa', 'RMSE_Dengan'}
if not required_cols.issubset(df_ablation.columns):
    raise KeyError(f'df_ablation harus berisi kolom: {required_cols}')

df_xgb = df_ablation[df_ablation['Model'] == 'XGBoost'].reset_index(drop=True)
df_lstm = df_ablation[df_ablation['Model'] == 'LSTM'].reset_index(drop=True)

# ============================================================
# GAMBAR 1 — XGBoost (file terpisah)
# ============================================================
emiten_xgb = df_xgb['Emiten'].tolist()
x = np.arange(len(emiten_xgb)); w = 0.35

fig, ax = plt.subplots(figsize=(3.5, 2.6))
ax.bar(x - w/2, df_xgb['RMSE_Tanpa'], w, label='X1 saja', color='#94A3B8')
ax.bar(x + w/2, df_xgb['RMSE_Dengan'], w, label='X1 + X2', color='#2DD4BF')
ax.set_ylabel('RMSE')
ax.set_xticks(x); ax.set_xticklabels(emiten_xgb, rotation=45, ha='right')
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
path_xgb = DIR_GAMBAR / 'gambar_ablation_xgboost.png'
plt.savefig(path_xgb, dpi=800, bbox_inches='tight')
plt.show()
plt.close(fig)
print(f'💾 Tersimpan: {path_xgb.name}')

# ============================================================
# GAMBAR 2 — LSTM (file terpisah)
# ============================================================
if not df_lstm.empty:
    emiten_lstm = df_lstm['Emiten'].tolist()
    x = np.arange(len(emiten_lstm))

    fig, ax = plt.subplots(figsize=(3.5, 2.6))
    ax.bar(x - w/2, df_lstm['RMSE_Tanpa'], w, label='X1 saja', color='#94A3B8')
    ax.bar(x + w/2, df_lstm['RMSE_Dengan'], w, label='X1 + X2', color='#EF4444')
    ax.set_ylabel('RMSE')
    ax.set_xticks(x); ax.set_xticklabels(emiten_lstm, rotation=45, ha='right')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)

    plt.tight_layout()
    path_lstm = DIR_GAMBAR / 'gambar_ablation_lstm.png'
    plt.savefig(path_lstm, dpi=800, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    print(f'💾 Tersimpan: {path_lstm.name}')
ax.bar(x + w/2, df_xgb['RMSE_Dengan'], w, label='X1 + X2', color='#2DD4BF')
ax.set_ylabel('RMSE')
ax.set_xticks(x); ax.set_xticklabels(emiten_list, rotation=45, ha='right')
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
path_xgb = DIR_GAMBAR / 'gambar_ablation_xgboost.png'
plt.savefig(path_xgb, dpi=800, bbox_inches='tight')
plt.show()
plt.close(fig)
print(f'💾 Tersimpan: {path_xgb.name}')

# ============================================================
# GAMBAR 2 — LSTM (file terpisah)
# ============================================================
if df_ablation['LSTM_RMSE_X1'].notna().any():
    fig, ax = plt.subplots(figsize=(3.5, 2.6))
    ax.bar(x - w/2, df_ablation['LSTM_RMSE_X1'],   w, label='X1 saja', color='#94A3B8')
    ax.bar(x + w/2, df_ablation['LSTM_RMSE_X1X2'], w, label='X1 + X2', color='#EF4444')
    ax.set_ylabel('RMSE')
    ax.set_xticks(x); ax.set_xticklabels(emiten_list, rotation=45, ha='right')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)

    plt.tight_layout()
    path_lstm = DIR_GAMBAR / 'gambar_ablation_lstm.png'
    plt.savefig(path_lstm, dpi=800, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    print(f'💾 Tersimpan: {path_lstm.name}')

---
## CELL 9 — Kesimpulan Ablation Study

In [ ]:
print('=' * 75)
print('  KESIMPULAN ABLATION STUDY')
print('=' * 75)

n_xgb_membaik = (df_ablation['XGB_Delta(%)'] > 0).sum()
n_lstm_membaik = (df_ablation['LSTM_Delta(%)'] > 0).sum() if df_ablation['LSTM_Delta(%)'].notna().any() else 0

print(f"XGBoost membaik dgn sentimen di {n_xgb_membaik}/{len(df_ablation)} emiten "
      f"(rata-rata delta RMSE: {df_ablation['XGB_Delta(%)'].mean():+.2f}%)")
if df_ablation['LSTM_Delta(%)'].notna().any():
    print(f"LSTM membaik dgn sentimen di {n_lstm_membaik}/{len(df_ablation)} emiten "
          f"(rata-rata delta RMSE: {df_ablation['LSTM_Delta(%)'].mean():+.2f}%)")

print()
print(f"Diebold-Mariano: signifikan (p<0.05) di {n_sig_xgb}/{len(df_dm)} emiten (XGBoost)")
if 'n_sig_lstm' in dir():
    print(f"                 signifikan (p<0.05) di {n_sig_lstm}/{len(df_dm)} emiten (LSTM)")

print()
print('PANDUAN PENULISAN ABSTRAK/BAB 4-5:')
if n_sig_xgb > 0 or ('n_sig_lstm' in dir() and n_sig_lstm > 0):
    print('-> ADA bukti statistik (DM test, p<0.05) di sebagian emiten -- klaim')
    print('   "sentimen berkontribusi signifikan" BISA dipertahankan, TAPI harus')
    print('   disebutkan eksplisit DI EMITEN MANA SAJA signifikan (bukan generalisasi')
    print('   ke seluruh 7 emiten kalau faktanya cuma sebagian).')
else:
    print('-> TIDAK ADA emiten dengan p<0.05 -- klaim "membuktikan signifikansi"')
    print('   TIDAK DAPAT dipertahankan secara statistik. Sarankan ubah bahasa abstrak')
    print('   jadi "mengidentifikasi kontribusi/peran sentimen" (deskriptif, bukan klaim')
    print('   signifikansi formal) -- kontribusi tetap bisa didukung SHAP/feature importance')
    print('   sebagai bukti kualitatif, bukan uji hipotesis kuantitatif.')
